# Import libraries and environment keys

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Helper functions
Used to load images as base64 strings

In [3]:
import base64
from pathlib import Path
import mimetypes

def load_image_as_base64(path: str) -> tuple[str, str]:
    """Return (base64_data, mime_type) for an image file."""
    image_bytes = Path(path).read_bytes()
    mime = mimetypes.guess_type(path)[0] or "image/png"
    return base64.b64encode(image_bytes).decode("utf-8"), mime

def load_image_from_path(path: str) -> bytes:
    """Return image bytes for an image file."""
    with open(path, "rb") as image_file:
        return image_file.read()


# Native Client Integrations

## Amazon and Anthropic Models




### Amazon Bedrock Models - text only

In [4]:
from gen_ai_hub.proxy.native.amazon.clients import Session

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "anthropic--claude-4-sonnet"

# Create a session with the model
bedrock = Session().client(model_name=model)
messages = [
    {
        "role": "user",
        "content": [
            {
                "text": "What is the capital of France?"
            }
        ],
    }
]
response = bedrock.converse(
    messages=messages,
    inferenceConfig={"maxTokens": max_Tokens, "temperature": temperature},
)
print(response)


{'ResponseMetadata': {'RequestId': '4bccd6e0-88af-426d-a6bc-9c6e7cbda180', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 05 Feb 2026 08:19:57 GMT', 'content-type': 'application/json', 'content-length': '345', 'x-aicore-request-id': 'b948759a-2a5c-9e09-b8cd-ffff242cd5ed', 'x-amzn-requestid': '4bccd6e0-88af-426d-a6bc-9c6e7cbda180', 'x-upstream-service-time': '1608'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': 'The capital of France is Paris.'}]}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 14, 'outputTokens': 10, 'totalTokens': 24, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}, 'metrics': {'latencyMs': 1452}}


In [5]:
print(response['output']['message']['content'][0]['text'])

The capital of France is Paris.


### Amazon Bedrock Models - text and images

In [6]:
from gen_ai_hub.proxy.native.amazon.clients import Session

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "anthropic--claude-4-sonnet"

# Create a session with the model
bedrock = Session().client(model_name=model)

# Load image and convert to base64
image_path = "SAP_logo.png"
fmt = "png"  # Format of the image
image_data = load_image_from_path(image_path)


messages = [
    {
        "role": "user",
        "content": [
            {
                "text": "What is the content of the image?"
            },
            {
                "image": {
                    "format": fmt, "source":{"bytes": image_data}
                }
            }
        ]
    }
]

response = bedrock.converse(
    messages=messages,
    inferenceConfig={"maxTokens": max_Tokens, "temperature": temperature},
)
print(response)

{'ResponseMetadata': {'RequestId': '46f8dfaf-e4b7-4856-9ca5-b2ee2c3bfe4e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 05 Feb 2026 08:20:04 GMT', 'content-type': 'application/json', 'content-length': '845', 'x-aicore-request-id': '39dee25c-c477-9fba-b14b-14f35a9a19a6', 'x-amzn-requestid': '46f8dfaf-e4b7-4856-9ca5-b2ee2c3bfe4e', 'x-upstream-service-time': '5734'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': 'This image shows the SAP logo on a grid background. The logo consists of the letters "SAP" in large white text displayed on a blue geometric shape. The blue shape appears to be a parallelogram or slanted rectangle that creates a dynamic, forward-leaning appearance. The background has a light gray grid pattern, suggesting this might be from a design or presentation context. SAP is a well-known German multinational software corporation that creates enterprise software to manage business operations and customer relations.'}]}}, 'stopRe

In [7]:
print(response['output']['message']['content'][0]['text'])

This image shows the SAP logo on a grid background. The logo consists of the letters "SAP" in large white text displayed on a blue geometric shape. The blue shape appears to be a parallelogram or slanted rectangle that creates a dynamic, forward-leaning appearance. The background has a light gray grid pattern, suggesting this might be from a design or presentation context. SAP is a well-known German multinational software corporation that creates enterprise software to manage business operations and customer relations.


## OpenAI Models



### OpenAI Models - text only

In [8]:
from gen_ai_hub.proxy.native.openai import chat

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "gpt-4o"  # Also compatible with Meta models like meta-llama3.1-70b-instruct
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

# Create a session with the model
response = chat.completions.create(messages=messages, model=model, temperature=temperature, max_tokens=max_Tokens)
print(response)


ChatCompletion(id='chatcmpl-D5ospZqvLvUiSXDIDyUzfpVDolQYn', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1770279607, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_ee1d74bde0', usage=CompletionUsage(completion_tokens=8, prompt_tokens=24, total_tokens=32, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'content_filter_res

In [9]:
print(response.choices[0].message.content)
usage = getattr(response, "usage", None)
if usage:
    print(f"Prompt tokens: {usage.prompt_tokens}")
    print(f"Completion tokens: {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")

The capital of France is Paris.
Prompt tokens: 24
Completion tokens: 8
Total tokens: 32


### OpenAI Models - text and images

In [10]:
model = "gpt-4o"
image_path = "SAP_logo.png"
base64_data, mime_type = load_image_as_base64(image_path)
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that can answer questions and help with tasks."
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "What is the content of the image?"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{base64_data}"
                }
            }
        ]
    }
]


response = chat.completions.create(
    messages=messages,
    model=model,
    temperature=temperature,
)

print(response.choices[0].message.content)
usage = getattr(response, "usage", None)
if usage:
    print(f"Prompt tokens: {usage.prompt_tokens}")
    

The image contains the logo of SAP, a multinational software corporation known for its enterprise resource planning (ERP) software. The logo features the letters "SAP" in white on a blue and cyan gradient background.
Prompt tokens: 1138


### OpenAI Models - Reasoning models
For reasoning tasks, we can define the "thinking budget"

In [11]:
# Model parameters
model = "gpt-5" 
messages = [
    {"role": "user", "content": "What is Green theorem?"}
]

# Create a session with the model
import time

# List of reasoning efforts
reasoning_efforts = ["minimal", "low", "medium", "high"]
responses = {}

# Time the response for each reasoning effort
for effort in reasoning_efforts:
    start_time = time.time()
    response = chat.completions.create(messages=messages, model=model, reasoning_effort=effort)  # OpenAI reasoning models do not allow temperature or max_tokens settings
    end_time = time.time()
    elapsed = end_time - start_time
    responses[effort] = {
        "response": response,
        "time_seconds": elapsed
    }
    print(f"Reasoning effort: {effort}, Time taken: {elapsed:.3f} seconds")
print(response.choices[0].message.content[:300] + '...')


Reasoning effort: minimal, Time taken: 3.756 seconds
Reasoning effort: low, Time taken: 4.372 seconds
Reasoning effort: medium, Time taken: 14.574 seconds
Reasoning effort: high, Time taken: 28.860 seconds
Green’s theorem relates a line integral around a simple closed curve in the plane to a double integral over the region it encloses.

Statement: Let C be a positively oriented (counterclockwise), piecewise smooth, simple closed curve in the plane, and let D be the region it bounds. If P(x,y) and Q(x,...


# Google Vertex AI Models

### Google Vertex AI Models - text only

In [21]:
from gen_ai_hub.proxy.native.google_genai.clients import Client
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')
client = Client(proxy_client=proxy_client)

response = client.models.generate_content(model="gemini-2.5-flash",
    contents="What is the capital of France?"
)

print(response.text)


The capital of France is **Paris**.


### Google Vertex AI Models - text and images

In [22]:
from gen_ai_hub.proxy.native.google_genai.clients import Client
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')
client = Client(proxy_client=proxy_client)

model_name = "gemini-2.5-flash"
image_path = "SAP_logo.png"
base64_data, mime_type = load_image_as_base64(image_path)

contents = [
    {
        "role": "user",
        "parts": [
            {"text": "What is the content of the image?"},
            {"inline_data": {"mime_type": mime_type, "data": base64_data}}
        ] 
    }
]


response = client.models.generate_content(model=model_name, contents=contents)
print(response.text)



The image displays the **SAP logo** set against a technical-style background.

Here's a breakdown of its content:

1.  **SAP Logo:**
    *   The prominent feature is the word "**SAP**" in large, bold, white, sans-serif capital letters.
    *   These letters are placed on a **blue rectangular background** that features a **gradient**. The gradient transitions from a lighter, brighter blue in the top-left area to a deeper, darker blue towards the bottom-right.
    *   A distinctive characteristic of this blue background is its **top-right corner, which is cut diagonally**, forming a triangular shape that points upwards and to the right.

2.  **Background Elements:**
    *   The entire logo is presented on a **light gray background** with a **fine grid pattern** resembling graph paper or a technical drafting sheet.
    *   Along the very top and bottom edges of the image, there are **subtle ruler markings**, further reinforcing the impression of a design or measurement context.


## Google Vertex AI Models - Multi-modal models
Gemini models can use as input not only text and images, but also audio and video, together with text.

In [23]:
from gen_ai_hub.proxy.native.google_genai.clients import Client
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

import base64
model_name = "gemini-2.5-flash"

# Load the media file
media_file = open("output.mp4", "rb")
encoded_media = base64.b64encode(media_file.read()).decode("utf-8")

# Detect the MIME type of the media file
def get_mime_type(file_path: str) -> str:
    """Determine MIME type based on file extension."""
    extension = file_path.lower().split('.')[-1]
    
    mime_types = {
        'mp4': 'video/mp4',
        'avi': 'video/avi', 
        'mov': 'video/mov',
        'webm': 'video/webm',
        'mp3': 'audio/mpeg',
        'wav': 'audio/wav',
        'flac': 'audio/flac',
        'm4a': 'audio/mp4',
        'ogg': 'audio/ogg'
    }
    
    return mime_types.get(extension, f'application/{extension}')

mime_type = get_mime_type("output.mp4")

proxy_client = get_proxy_client('gen-ai-hub')
client = Client(proxy_client=proxy_client)

contents = [
    {
        "role": "user",
        "parts": [
            {"text": "1. Are there any safety violations in the video? 2. Are the railings visible on the stairs? If not, is it dangerous? 3. What safety measures should be taken based on what I saw? 4. List all safety violation and seconds"},
            {"inline_data": {"mime_type": mime_type, "data": encoded_media}}
        ] 
    }
]


response = client.models.generate_content(model=model_name, contents=contents)
print(response.text)

Based on the video provided, here's an analysis of the safety aspects:

**1. Are there any safety violations in the video?**
Yes, there are several significant safety violations visible in the video, primarily related to fall protection, housekeeping, and potentially personal protective equipment (PPE).

**2. Are the railings visible on the stairs? If not, is it dangerous?**
No, proper railings are **not visible** on the stairs leading up to the elevated platform. There are some vertical supports on the right side, but no continuous handrail or guardrail system on either side. This is **extremely dangerous**. The absence of railings creates a severe fall hazard, especially when ascending or descending with tools or materials, or if the surface is slippery.

**3. What safety measures should be taken based on what I saw?**

*   **Fall Protection:**
    *   **Install proper guardrails:** All elevated platforms, including the drilling rig platform where workers are handling pipes, must hav